# MCBOMs Harwood (2003) Case StudyReproduces the network-level optimization from Harwood et al. (2003) at the$50M unconstrained budget. The published reference values are:| Metric | Value ||---|---|| Resurfacing cost | $11.79 M || Safety cost | $4.48 M || PSB | $9.83 M || PTOB | $0.81 M || Net benefit | $6.16 M |(Harwood et al., 2003, Table 4; values in 2002 dollars.)

## 1. Install MCBOMsThe framework installs from the public TTI repository.

In [ ]:
!pip install --quiet 'git+https://github.com/ttitamu/mcboms-optimization.git#egg=mcboms[pulp]'

## 2. Load the Harwood dataThe 10 sites and their alternatives are taken from Tables 2 and 3 of theHarwood paper. The loader functions ship with the package.

In [ ]:
from mcboms.io.readers import load_harwood_sitesfrom mcboms.data.harwood_alternatives import get_harwood_alternativessites = load_harwood_sites()alternatives = get_harwood_alternatives()print(f"Sites:        {len(sites)}")print(f"Alternatives: {len(alternatives)}")print(f"Avg alts/site: {len(alternatives) / len(sites):.1f}")

## 3. Solve at the $50M budget

In [ ]:
from mcboms.core.optimizer import Optimizeropt = Optimizer(sites=sites, alternatives=alternatives, budget=50_000_000)result = opt.solve()  # auto-detects gurobi or pulpprint(result.summary())

## 4. Compare against published valuesHarwood Table 4 reports the breakdown for the $50M case. The MCBOMs resultshould match within rounding tolerance.

In [ ]:
harwood_table4 = {    "resurfacing_cost":  11_790_000,    "safety_cost":        4_480_000,    "PSB":                9_830_000,    "PTOB":                 810_000,    "net_benefit":        6_160_000,}# Aggregate from the result.sel = result.selected_alternativesmcboms_resurfacing = sel["resurfacing_cost"].sum() if "resurfacing_cost" in sel.columns else 0mcboms_safety_cost = sel["safety_improvement_cost"].sum() if "safety_improvement_cost" in sel.columns else 0mcboms_safety_benefit = sel["safety_benefit"].sum() if "safety_benefit" in sel.columns else 0mcboms_ops_benefit = sel["ops_benefit"].sum() if "ops_benefit" in sel.columns else 0import pandas as pdcomparison = pd.DataFrame({    "Harwood Table 4": [        harwood_table4["resurfacing_cost"],        harwood_table4["safety_cost"],        harwood_table4["PSB"],        harwood_table4["PTOB"],        harwood_table4["net_benefit"],    ],    "MCBOMs": [        mcboms_resurfacing,        mcboms_safety_cost,        mcboms_safety_benefit,        mcboms_ops_benefit,        result.net_benefit,    ],}, index=["Resurfacing cost", "Safety cost", "PSB", "PTOB", "Net benefit"])comparison["Δ %"] = ((comparison["MCBOMs"] - comparison["Harwood Table 4"])                    / comparison["Harwood Table 4"] * 100).round(1)# Format for display.for col in ("Harwood Table 4", "MCBOMs"):    comparison[col] = comparison[col].apply(lambda v: f"${v:,.0f}")comparison

## 5. Selected sitesThe Harwood paper's Table 2 lists the 10 sites and their selected alternativesat the $50M budget. The MCBOMs solution should match.

In [ ]:
result.selected_alternatives[["site_id", "alt_id", "description"]]